# Molecular Property Predictions — Variant × Training Fraction Analysis

Compares models across two graph-construction strategies:
- **cutoff** (`experiments/molecular`): ScalarGNNMolecular, 3.5 Bohr cutoff + 4 NN — usable downstream on full QM9. Fractions: 0.01, 0.05, 0.1, 1.0.
- **cp** (`experiments/aimel_cp_molecular`): ScalarTPaiNNMolecular, BCP/RCP connectivity — *not* usable downstream. Fractions: 0.1, 0.5, 1.0.

Both use `informed` (atomic AIM features) and `blind` (no atomic priors) variants.

In [ ]:
import importlib
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath(os.path.join('..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import analysis_molecular as mol
importlib.reload(mol)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# ── Paths and configuration ───────────────────────────────────────────────────
EXPERIMENT_DIR_CUTOFF = os.path.join(project_root, 'experiments', 'molecular')

VARIANTS      = mol.DEFAULT_VARIANTS          # ['informed', 'blind']
FRACS_CUTOFF  = [0.01, 0.05, 0.1, 1.0]       # cutoff-graph experiment fractions
PROPS         = mol.MOLECULAR_PROPS           # ['alpha', 'gap', 'U0', 'Cv']

# Model lists
MODELS_CUTOFF = [f'cutoff_{v}_{f}' for v in VARIANTS for f in FRACS_CUTOFF]
MODELS_ALL    = MODELS_CUTOFF 

# Prefixes for learning curves (one line per prefix across its fractions)
PREFIXES_CUTOFF = [f'cutoff_{v}' for v in VARIANTS]

# Colours per prefix
MODEL_COLORS = {
    'cutoff_informed': '#d62728',  
    'cutoff_blind':    '#1f77b4',   
}

print('cutoff models:', MODELS_CUTOFF)

## Load metrics

In [ ]:
mol_cutoff = mol.precompute_metrics_mol(
    experiment_dir=EXPERIMENT_DIR_CUTOFF,
    experiment_tag='cutoff',
    variants=VARIANTS, fractions=FRACS_CUTOFF, properties=PROPS,
)

## Summary tables

In [ ]:
print('=== cutoff ===')
mol.metric_summary_mol(mol_cutoff, models=MODELS_CUTOFF, metric='R2', split='test')

## Learning curves

Each experiment is plotted with its own fraction axis. Shared fractions (0.1 and 1.0) allow direct comparison.

In [ ]:
# Cutoff: 4 fractions
import importlib
import analysis_molecular as mol
importlib.reload(mol)
mol.learning_curve_mol(
    metric='R2', models=MODELS_CUTOFF, mol_metrics=mol_cutoff,
    split='test', model_prefixes=PREFIXES_CUTOFF, fractions=FRACS_CUTOFF,
    model_colors={p: MODEL_COLORS[p] for p in PREFIXES_CUTOFF},
    save_path='learning_curve_all_R2_test.pdf',
)

## Informed vs Blind — paired test per (fraction, property)

At each training fraction we compare Informed and Blind on the same 25
CV folds, so a **paired** test is appropriate. We report the paired-$t$
p-value (assumption: differences approx. normal) and the Wilcoxon
signed-rank p-value (no normality required) for each QM property.
Equal variance between Informed and Blind is **not** required for paired
tests, so Levene-failure across training fractions is irrelevant here.


In [ ]:
importlib.reload(mol)
ivb_cutoff = mol.paired_informed_vs_blind_table(
    mol_cutoff,
    fractions=FRACS_CUTOFF,
    experiment_tag='cutoff',
    metric='R2',
    split='test',
)
ivb_cutoff


## Paired-test validity diagnostics

For each (fraction, property) we compute Shapiro–Wilk on the 25 paired
differences (the relevant assumption for the paired $t$-test) and on
each model's 25 fold values (descriptive). Levene/homoscedasticity is
deliberately omitted: paired tests on matched folds do not require it.
Each fraction is a multirow over the test quantities; each property
spans a 2-column block (Informed | Blind).


In [ ]:
importlib.reload(mol)
paired_diag_cutoff = mol.paired_diagnostics_mol(
    mol_cutoff,
    fractions=FRACS_CUTOFF,
    experiment_tag='cutoff',
    metric='R2',
    split='test',
)
paired_diag_cutoff.round(4)


In [ ]:
_ = mol.latex_paired_diagnostics_mol(
    paired_diag_cutoff,
    fractions=FRACS_CUTOFF,
    family='cutoff',
    metric='R2',
    label='tab:paired_diag_cutoff_R2',
)
